### `Generate Key`

In [ ]:
import random
import string


def generate_key() -> str:
    length = random.randint(4, 8)
    return "".join(random.choices(string.ascii_uppercase + string.digits, k=length))


key = generate_key()
print(f"Generated key: {key}")

### `Encryption`

In [ ]:
import os
import json
import string

CHARS = string.ascii_uppercase + string.digits


def generate_key_table(key: str) -> list[list[str]]:
    key = key.upper()
    seen = []
    for ch in key:
        if ch in CHARS and ch not in seen:
            seen.append(ch)
    for ch in CHARS:
        if ch not in seen:
            seen.append(ch)
    return [seen[i * 6 : (i + 1) * 6] for i in range(6)]


def get_position(table, ch):
    for r, row in enumerate(table):
        if ch in row:
            return r, row.index(ch)
    raise ValueError(f"Character '{ch}' not in table.")


def prepare_alpha(alpha_str: str) -> tuple[list[str], list[int]]:
    filtered = list(alpha_str)
    digrams = []
    filler_positions = []
    i = 0
    while i < len(filtered):
        a = filtered[i]
        if i + 1 < len(filtered):
            b = filtered[i + 1]
            if a == b:
                filler = "9" if a == "X" else "X"
                digrams.append(a)
                filler_positions.append(len(digrams))
                digrams.append(filler)
                i += 1
            else:
                digrams.extend([a, b])
                i += 2
        else:
            filler = "9" if a == "X" else "X"
            digrams.append(a)
            filler_positions.append(len(digrams))
            digrams.append(filler)
            i += 1
    return digrams, filler_positions


def encrypt_digrams(table, digrams: list[str]) -> str:
    result = []
    for i in range(0, len(digrams), 2):
        a, b = digrams[i], digrams[i + 1]
        ra, ca = get_position(table, a)
        rb, cb = get_position(table, b)
        if ra == rb:
            result.append(table[ra][(ca + 1) % 6])
            result.append(table[rb][(cb + 1) % 6])
        elif ca == cb:
            result.append(table[(ra + 1) % 6][ca])
            result.append(table[(rb + 1) % 6][cb])
        else:
            result.append(table[ra][cb])
            result.append(table[rb][ca])
    return "".join(result)


def encrypt(plaintext: str, key: str) -> tuple[str, dict]:
    table = generate_key_table(key)
    upper = plaintext.upper()

    non_alpha = {}
    alpha_chars = []
    alpha_positions = []

    for i, ch in enumerate(upper):
        if ch in CHARS:
            alpha_chars.append(ch)
            alpha_positions.append(i)
        else:
            non_alpha[str(i)] = plaintext[i]

    digrams, filler_positions = prepare_alpha("".join(alpha_chars))
    encrypted_body = encrypt_digrams(table, digrams)

    meta = {
        "non_alpha": non_alpha,
        "alpha_positions": alpha_positions,
        "filler_positions": filler_positions,
        "original_length": len(plaintext),
    }

    return encrypted_body, meta


file_path = input("Enter path of plain text file: ").strip()
key = input("Enter key value: ").strip()

if not os.path.exists(file_path):
    print(f"Error: File '{file_path}' not found.")
else:
    with open(file_path, "r", encoding="utf-8") as f:
        plain_text = f.read()

    encrypted_text, meta = encrypt(plain_text, key)

    file_name = os.path.splitext(os.path.basename(file_path))[0]
    directory = os.path.dirname(file_path)
    output_file = os.path.join(directory, f"{file_name}_Encrypted_PF.txt")
    meta_file = os.path.join(directory, f"{file_name}_Meta_PF.json")

    with open(output_file, "w", encoding="utf-8") as f:
        f.write(encrypted_text)

    with open(meta_file, "w", encoding="utf-8") as f:
        json.dump(meta, f)

    print(f"Encryption successful!")
    print(f"Key used: {key}")
    print(f"Original file: {file_path}")
    print(f"Encrypted file saved: {output_file}")
    print(f"Meta file saved: {meta_file}")

### `Decryption`

In [ ]:
import os
import json
import string

CHARS = string.ascii_uppercase + string.digits


def generate_key_table(key: str) -> list[list[str]]:
    key = key.upper()
    seen = []
    for ch in key:
        if ch in CHARS and ch not in seen:
            seen.append(ch)
    for ch in CHARS:
        if ch not in seen:
            seen.append(ch)
    return [seen[i * 6 : (i + 1) * 6] for i in range(6)]


def get_position(table, ch):
    for r, row in enumerate(table):
        if ch in row:
            return r, row.index(ch)
    raise ValueError(f"Character '{ch}' not in table.")


def decrypt_digrams(table, ciphertext: str) -> str:
    chars = [ch for ch in ciphertext.upper() if ch in CHARS]
    if len(chars) % 2 != 0:
        raise ValueError("Ciphertext has odd number of valid characters.")
    result = []
    for i in range(0, len(chars), 2):
        a, b = chars[i], chars[i + 1]
        ra, ca = get_position(table, a)
        rb, cb = get_position(table, b)
        if ra == rb:
            result.append(table[ra][(ca - 1) % 6])
            result.append(table[rb][(cb - 1) % 6])
        elif ca == cb:
            result.append(table[(ra - 1) % 6][ca])
            result.append(table[(rb - 1) % 6][cb])
        else:
            result.append(table[ra][cb])
            result.append(table[rb][ca])
    return "".join(result)


def remove_fillers(
    decrypted_chars: list[str], filler_positions: list[int]
) -> list[str]:
    result = []
    filler_set = set(filler_positions)
    for i, ch in enumerate(decrypted_chars):
        if i not in filler_set:
            result.append(ch)
    return result


def decrypt(ciphertext: str, key: str, meta: dict) -> str:
    table = generate_key_table(key)

    raw_decrypted = decrypt_digrams(table, ciphertext)
    raw_list = list(raw_decrypted)

    filler_positions = meta["filler_positions"]
    clean_alpha = remove_fillers(raw_list, filler_positions)

    non_alpha = meta["non_alpha"]
    original_length = meta["original_length"]
    alpha_positions = meta["alpha_positions"]

    output = [""] * original_length

    for idx_str, ch in non_alpha.items():
        output[int(idx_str)] = ch

    for i, orig_idx in enumerate(alpha_positions):
        if i < len(clean_alpha):
            output[orig_idx] = clean_alpha[i]
        else:
            output[orig_idx] = "?"

    return "".join(output)


file_path = input("Enter path of encrypted text file: ").strip()
key = input("Enter key value: ").strip()
meta_path = input("Enter path of meta JSON file: ").strip()

if not os.path.exists(file_path):
    print(f"Error: File '{file_path}' not found.")
elif not os.path.exists(meta_path):
    print(f"Error: Meta file '{meta_path}' not found.")
else:
    with open(file_path, "r", encoding="utf-8") as f:
        encrypted_text = f.read()

    with open(meta_path, "r", encoding="utf-8") as f:
        meta = json.load(f)

    decrypted_text = decrypt(encrypted_text, key, meta)

    file_name = os.path.splitext(os.path.basename(file_path))[0]
    directory = os.path.dirname(file_path)
    output_file = os.path.join(directory, f"{file_name}_Decrypted_PF.txt")

    with open(output_file, "w", encoding="utf-8") as f:
        f.write(decrypted_text)

    print(f"Decryption successful!")
    print(f"Key used: {key}")
    print(f"Encrypted file: {file_path}")
    print(f"Decrypted file saved: {output_file}")